# Deep Convolutional GAN (DCGAN) on CelebA Face Dataset 🎨👤

This notebook implements and trains a **Deep Convolutional Generative Adversarial Network (DCGAN)**
to synthesize realistic human face images from 100-dimensional random Gaussian noise vectors $z \sim \mathcal{N}(0, 1)$.

### Key Objectives:
1. **CelebA Preprocessing**: Crop, resize to $64 \times 64$, and normalize to $[-1, 1]$.
2. **Generator Architecture**: Transposed convolutions mapping latent vectors to RGB images.
3. **Discriminator Architecture**: Strided convolutions classifying real vs synthetic images.
4. **Adversarial Training**: Two-player minimax game using binary cross-entropy loss.
5. **Latent Space Exploration**: Linear manifold interpolation between distinct faces.


In [ ]:
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from pathlib import Path

from models import Generator, Discriminator, weights_init
from dataset import get_celeba_dataloader, denormalize
from train import train_dcgan

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on compute device: {device}")


## 1. Inspecting Real CelebA Dataset Batches
Displaying a batch of ground truth CelebA human faces after preprocessing.


In [ ]:
dataloader = get_celeba_dataloader('data/celeba/img_align_celeba', batch_size=16, image_size=64)
real_batch, _ = next(iter(dataloader))

plt.figure(figsize=(8, 8))
grid = vutils.make_grid(denormalize(real_batch), nrow=4, padding=2)
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.title('Sample Preprocessed CelebA Training Images (64x64)')
plt.axis('off')
plt.show()


## 2. Adversarial Training Loop Execution
Training Generator and Discriminator iteratively over multiple epochs.


In [ ]:
report = train_dcgan(
    data_dir='data/celeba/img_align_celeba',
    output_dir='checkpoints',
    samples_dir='samples',
    num_epochs=4,
    batch_size=16,
    device=str(device)
)
print('Training summary:', report)


## 3. Loss Dynamics Visualization
Tracking Generator vs. Discriminator Minimax loss convergence.


In [ ]:
hist = report['history']
plt.figure(figsize=(10, 4))
plt.plot(hist['G_losses'], label='Generator Loss', color='#2563EB', lw=2)
plt.plot(hist['D_losses'], label='Discriminator Loss', color='#DC2626', lw=2)
plt.title('DCGAN Adversarial Training Dynamics')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


## 4. Synthesizing High-Quality Synthetic Faces
Generating novel, non-existent human faces from random latent noise vectors.


In [ ]:
netG = Generator(nz=100, ngf=64, nc=3).to(device)
netG.load_state_dict(torch.load('checkpoints/generator.pth', map_location=device, weights_only=True))
netG.eval()

torch.manual_seed(123)
test_noise = torch.randn(16, 100, 1, 1, device=device)
with torch.no_grad():
    fake_faces = netG(test_noise).detach().cpu()

plt.figure(figsize=(8, 8))
fake_grid = vutils.make_grid(denormalize(fake_faces), nrow=4, padding=2)
plt.imshow(fake_grid.permute(1, 2, 0).numpy())
plt.title('Synthetic Human Faces Generated by DCGAN')
plt.axis('off')
plt.show()


## 5. Latent Space Linear Interpolation (Face Morphing)
Walking smoothly between two distinct face vectors $z_A$ and $z_B$ in the learned continuous manifold:
$$z(\alpha) = (1 - \alpha) z_A + \alpha z_B$$


In [ ]:
import numpy as np
torch.manual_seed(42)
z_a = torch.randn(1, 100, 1, 1, device=device)
torch.manual_seed(99)
z_b = torch.randn(1, 100, 1, 1, device=device)

alphas = np.linspace(0.0, 1.0, 8)
interpolated = torch.cat([((1 - a) * z_a + a * z_b) for a in alphas], dim=0)

with torch.no_grad():
    morph = netG(interpolated).detach().cpu()

plt.figure(figsize=(14, 3))
morph_grid = vutils.make_grid(denormalize(morph), nrow=8, padding=2)
plt.imshow(morph_grid.permute(1, 2, 0).numpy())
plt.title('Smooth Latent Space Interpolation: Face A (Left) -> Face B (Right)')
plt.axis('off')
plt.show()
